### Pipeline 3 — Neural Network (MLPClassifier)

In [1]:
import os
import pandas as pd

# --- Path Config ---
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
STATS_DIR = os.path.join(OUTPUTS_DIR, "statistics")
PLOTS_DIR = os.path.join(OUTPUTS_DIR, "plots")
METRICS_DIR = os.path.join(OUTPUTS_DIR, "metrics")

# Make sure subfolders exist
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

part12_metrics_dir = os.path.join(METRICS_DIR, "part12_neural_models")
os.makedirs(part12_metrics_dir, exist_ok=True)



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def train_tf_mlp(filepath, target_col, handle_unknown=True, sampler_type="SMOTE",
                 epochs=30, batch_size=64, output_dir="Results_TF"):

    print(f"\n Training TensorFlow MLP for: {target_col} using {sampler_type}")

    # Load and clean
    df = pd.read_csv(filepath)
    if handle_unknown:
        df = df[df[target_col] != "Unknown"]

    y = df[target_col].apply(lambda x: 1 if x == "Yes" else 0)
    X = df.drop(columns=[target_col])

    # Train-validation split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    # One-hot encode categorical features
    cat_cols = X.select_dtypes(include="object").columns.tolist()
    preprocessor = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
    ], remainder="passthrough")

    X_train_encoded = preprocessor.fit_transform(X_train)
    X_val_encoded = preprocessor.transform(X_val)

    # Apply balancing
    if sampler_type == "SMOTE":
        X_train_encoded, y_train = SMOTE(random_state=42).fit_resample(X_train_encoded, y_train)
    elif sampler_type == "undersample":
        X_train_encoded, y_train = RandomUnderSampler(random_state=42).fit_resample(X_train_encoded, y_train)

    # Create output dir
    out_path = os.path.join(output_dir, target_col.replace(" ", "").replace("(", "").replace(")", ""), sampler_type)
    os.makedirs(out_path, exist_ok=True)

    # Build model
    model = Sequential([
        Dense(64, activation="relu", input_shape=(X_train_encoded.shape[1],)),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        loss="binary_crossentropy",
        optimizer="adam",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )

    early_stop = EarlyStopping(patience=5, restore_best_weights=True)
    history = model.fit(
        X_train_encoded, y_train,
        validation_data=(X_val_encoded, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    # === Plot Loss & Accuracy ===
    plt.figure()
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Val Loss")
    plt.title("Loss Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(out_path, "loss_curve.png"))
    plt.close()

    plt.figure()
    plt.plot(history.history["accuracy"], label="Train Acc")
    plt.plot(history.history["val_accuracy"], label="Val Acc")
    plt.title("Accuracy Curve")
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(out_path, "accuracy_curve.png"))
    plt.close()

    # === Evaluation ===
    y_pred_proba = model.predict(X_val_encoded).ravel()
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc_score = roc_auc_score(y_val, y_pred_proba)
    fpr, tpr, _ = roc_curve(y_val, y_pred_proba)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(out_path, "roc_curve.png"))
    plt.close()

    cm = confusion_matrix(y_val, y_pred)
    report = classification_report(y_val, y_pred, output_dict=True)

    # Save evaluation outputs
    pd.DataFrame(report).transpose().to_csv(os.path.join(out_path, "classification_report.csv"))
    pd.DataFrame(cm, index=["True 0", "True 1"], columns=["Pred 0", "Pred 1"]).to_csv(os.path.join(out_path, "confusion_matrix.csv"))

    plt.figure(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, "conf_matrix.png"))
    plt.close()

    # Display key metrics in console
    print("\n Key Evaluation Metrics:")
    for key in ["precision", "recall", "f1-score"]:
        print(f"Yes (1) {key.capitalize()}: {report['1'][key]:.4f}")
        print(f"No  (0) {key.capitalize()}: {report['0'][key]:.4f}")
    print(f"Macro F1: {report['macro avg']['f1-score']:.4f}")
    print(f"ROC AUC: {auc_score:.4f}")
    print(" All evaluation reports and plots saved.")


In [4]:
train_tf_mlp(
    filepath=os.path.join(STATS_DIR, "model1_high_bp.csv"),
    target_col="Has a high blood pressure",
    handle_unknown=False,
    sampler_type="SMOTE",
    output_dir=part12_metrics_dir
)

train_tf_mlp(
    filepath=os.path.join(STATS_DIR, "model2_diabetes.csv"),
    target_col="Has diabetes",
    handle_unknown=True,
    sampler_type="SMOTE",
    output_dir=part12_metrics_dir
)

train_tf_mlp(
    filepath=os.path.join(STATS_DIR, "model3_cardio.csv"),
    target_col="Cardiovascular condition (Heart disease or stroke)",
    handle_unknown=True,
    sampler_type="SMOTE",
    output_dir=part12_metrics_dir
)



 Training TensorFlow MLP for: Has a high blood pressure using SMOTE


d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.7187 - auc: 0.7884 - loss: 0.5485 - val_accuracy: 0.6850 - val_auc: 0.8080 - val_loss: 0.5513
Epoch 2/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7414 - auc: 0.8155 - loss: 0.5170 - val_accuracy: 0.7155 - val_auc: 0.8070 - val_loss: 0.5246
Epoch 3/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7452 - auc: 0.8207 - loss: 0.5114 - val_accuracy: 0.7074 - val_auc: 0.8040 - val_loss: 0.5285
Epoch 4/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7527 - auc: 0.8278 - loss: 0.5021 - val_accuracy: 0.7122 - val_auc: 0.8032 - val_loss: 0.5345
Epoch 5/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7568 - auc: 0.8323 - loss: 0.4969 - val_accuracy: 0.7099 - val_auc: 0.7988 - val_loss: 0.5358
Epoch 6/30
1811/1811 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7588 - auc: 0.8331 - loss: 0.4953 - val_accuracy: 0.7163 - val_auc: 0.8009 - val_loss: 0.5276
Epoch 7/30
1811/1811 ━━━━━━━

d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2234/2234 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7757 - auc: 0.8448 - loss: 0.4765 - val_accuracy: 0.7466 - val_auc: 0.8449 - val_loss: 0.4775
Epoch 2/30
2234/2234 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8117 - auc: 0.8808 - loss: 0.4206 - val_accuracy: 0.7452 - val_auc: 0.8341 - val_loss: 0.4708
Epoch 3/30
2234/2234 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8247 - auc: 0.8935 - loss: 0.3981 - val_accuracy: 0.7820 - val_auc: 0.8349 - val_loss: 0.4196
Epoch 4/30
2234/2234 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8328 - auc: 0.9016 - loss: 0.3821 - val_accuracy: 0.7800 - val_auc: 0.8288 - val_loss: 0.4242
Epoch 5/30
2234/2234 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8368 - auc: 0.9072 - loss: 0.3713 - val_accuracy: 0.7953 - val_auc: 0.8299 - val_loss: 0.3963
Epoch 6/30
2234/2234 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8406 - auc: 0.9115 - loss: 0.3633 - val_accuracy: 0.7945 - val_auc: 0.8309 - val_loss: 0.3922
Epoch 7/30
2234/2234 ━━━━━━━━━━━━━━━━━━

d:\Winter\DataScienceWorkplace\Basics\.venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2195/2195 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7847 - auc: 0.8569 - loss: 0.4597 - val_accuracy: 0.7717 - val_auc: 0.8491 - val_loss: 0.4166
Epoch 2/30
2195/2195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8165 - auc: 0.8892 - loss: 0.4065 - val_accuracy: 0.7489 - val_auc: 0.8413 - val_loss: 0.4720
Epoch 3/30
2195/2195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8269 - auc: 0.8992 - loss: 0.3878 - val_accuracy: 0.7911 - val_auc: 0.8408 - val_loss: 0.4078
Epoch 4/30
2195/2195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8378 - auc: 0.9075 - loss: 0.3705 - val_accuracy: 0.7963 - val_auc: 0.8371 - val_loss: 0.3896
Epoch 5/30
2195/2195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8386 - auc: 0.9107 - loss: 0.3649 - val_accuracy: 0.7861 - val_auc: 0.8364 - val_loss: 0.4139
Epoch 6/30
2195/2195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8433 - auc: 0.9135 - loss: 0.3583 - val_accuracy: 0.7824 - val_auc: 0.8341 - val_loss: 0.4269
Epoch 7/30
2195/2195 ━━━━━━━━━━━━━━━━━━

## Pipeline 3 Summary: TensorFlow MLP (SMOTE Oversampling)

We evaluated a Multi-Layer Perceptron (MLP) for all three target variables using:
- One-hot encoded categorical features
- SMOTE Oversampling for class balance
- Early stopping to prevent overfitting
- Full evaluation with ROC AUC, F1-score, and confusion matrix

### Results Overview

| Target                          | F1 (Yes) | ROC AUC | Comparison to Logistic Regression |
|----------------------------------|----------|---------|------------------------------------|
| **High Blood Pressure**          | 0.610    | 0.808   | Slightly lower F1, similar AUC     |
| **Diabetes**                     | 0.568    | 0.820   | Nearly identical                   |
| **Cardiovascular Condition**     | 0.593    | 0.809   | Slightly lower F1 and AUC          |

---

### Final Verdict:

While the MLP model performed reasonably well, it did **not outperform** our baseline:
- Logistic Regression (with SMOTE Oversampling) remains the **most reliable and interpretable model**.
- No major gains in F1-score or ROC AUC were observed with MLP.

### Decision:

We will **retain Logistic Regression with SMOTE Oversampling** as the final model across all targets:
-  Simpler and faster
-  Better interpretability
-  Comparable or superior performance

